# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All entities—including record sets, fields, and columns—are referenced by their `@id` values according to best practices.

### Dataset Source
The dataset is loaded from the following Croissant schema URL:

**https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json**

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and inspect the schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# URL to the Croissant schema JSON-LD
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset Croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a CroissantMetadata object, not a dict

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Let's review which **record sets** are available in this dataset, and show their fields and their `@id`s.

All tables/sheets that hold data are defined as **record sets**. Each field (column) of a table is described by a **field** object, and both have unique `@id` values.

In [ ]:
# List available record sets and their field @ids
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")

for i, record_set in enumerate(record_sets):
    print(f"Record Set {i+1}: @id = {record_set['@id']}")
    print(f"  name: {record_set.get('name','-')}\n  description: {record_set.get('description','-')}")
    # List field ids (columns)
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        # Single field case
        fields = [fields]
    print(f"  Fields/Columns:")
    for f in fields:
        # Each field is an @id or dict (if expanded)
        if isinstance(f, dict):
            print(f"    - @id: {f['@id']} name: {f.get('name','')}")
        elif isinstance(f, str):
            print(f"    - @id: {f}")
    print()

## 3. Data Extraction
Load the records from a selected record set into a pandas DataFrame.

⚠️ **IMPORTANT:**
- You must use record set and field `@id` values as they appear above to extract the correct data.
- The main data record set for the FAIR² dataset (usually proteomics data) is typically named `ProteinAbundance` or similar, but always select by the real `@id` found above.
- For illustration, we'll select the first record set found in the schema.

In [ ]:
# You should update this to the actual @id of the main data record set
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]['@id']
else:
    raise RuntimeError("No record sets found in the metadata.")

# For demonstration, load all record sets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records from record set {rsid}...")
        dataframes[rsid] = df
    except Exception as e:
        print(f"Failed to load records for {rsid}: {e}")

# Show the columns of the main record set
main_df = dataframes[main_record_set_id]
print(f"\nColumns in main record set ({main_record_set_id}):\n{main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's process the main DataFrame:
- Select a numeric field (by `@id` or DataFrame column name)
- Filter on that field
- Normalize it (Z-score)
- Optionally group by another field

In [ ]:
# --- Inspect to choose fields for EDA (edit to suit your data) ---
print("\nSample columns in the main record set:")
print(main_df.columns.tolist())

# You need to pick a numeric field for demo. Let's try to auto-detect one.
numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
if not numeric_fields:
    # Try parsing columns as float if they look numeric but are object dtype
    for col in main_df.columns:
        try:
            converted = pd.to_numeric(main_df[col], errors='coerce')
            if converted.notna().sum() > 0:
                main_df[col + "_NUM"] = converted
                numeric_fields.append(col + "_NUM")
        except Exception:
            pass

if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")
else:
    raise Exception("Could not identify a numeric field. Please manually specify one.")

# Use a threshold for filtering
threshold = main_df[numeric_field].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field]) else 10

filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by another field, e.g. the second non-numeric field
non_numeric_fields = [col for col in main_df.columns if not pd.api.types.is_numeric_dtype(main_df[col])]
group_field = non_numeric_fields[0] if len(non_numeric_fields) > 0 else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field} (showing means):")
    print(grouped_df.head())

## 5. Visualization
Let's plot the distribution of our chosen numeric field, and, if a group field was identified, visualize the grouped means.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field and grouped_df available, plot mean per group
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(12, 4))
    grouped_df[numeric_field].plot(kind='bar')
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(f"{group_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and process the FAIR² proteomics dataset using the `mlcroissant` library, referencing all data entities by their `@id` fields. We extracted data from record sets, performed basic filtering and normalization, and visualized the main quantitative distribution. This workflow provides a foundation for further advanced analysis, such as biomarker discovery, clustering, or ML modeling.

You can now extend this notebook by mapping additional record sets, exploring metadata, or building downstream pipelines using FAIR, reproducible standards.